In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ============================================================
# Zugriff auf Ordner und Datein durch Github
# ============================================================

import os
import sys
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"



token = getpass("GitHub token: ")


REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
Cloning into '/content/bachelor-thesis'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 129 (delta 45), reused 105 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 86.45 KiB | 6.65 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/bachelor-thesis
Repo exists: True


In [4]:
!git pull

Already up to date.


In [5]:
# ============================================================
# Imports & globale Konfiguration
# Imports & global configuration
# ============================================================


import sys
from pathlib import Path

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))

import numpy as np
import torch

from utils.dataset import TranADDataset, TranADDataLoader

from explainability.attention import TranADExplainability
from utils.seed import set_seed

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [6]:
# ============================================================
# Demo: ECG Daten
# Vollständiger Durchlauf / Complete end-to-end run
# Schritt / Step: Daten → DataLoader → Modell → Training →
#                 Erkennung → Evaluation → Explainability
# ============================================================

print("  TranAD Demo – ECG Data")


from pathlib import Path
import numpy as np

# Load my preprocessed data
# Using the same processed-data path as in my preprocessing notebook
PROCESSED_DIR = Path("/content/drive/MyDrive/ECG data/processed")

# Load already preprocessed and normalized ECG arrays
TrainingData = np.load(PROCESSED_DIR / "X_train_small.npy")
TestData = np.load(PROCESSED_DIR / "X_test_small.npy")
TestLabels = np.load(PROCESSED_DIR / "y_test_small.npy")

print("TrainingData:", TrainingData.shape)
print("TestData:", TestData.shape)
print("TestLabels:", TestLabels.shape)


  TranAD Demo – ECG Data
TrainingData: (600, 1000, 12)
TestData: (200, 1000, 12)
TestLabels: (200,)


In [8]:
# ── 2. TranADDataLoader erstellen / Create TranADDataLoader ─────────
NumFeatures = TrainingData.shape[-1]   # 12 ECG leads
WindowSize = TrainingData.shape[1]     # 1000 time points


DataManager = TranADDataLoader(
    TrainingData       = TrainingData,
    TestData           = TestData,
    WindowSize         = WindowSize,
    TrainingStride     = WindowSize,          # Überlappende Fenster beim Training / Overlapping training windows
    TestStride         = WindowSize,       # Standard: kein Überlapp (= WindowSize) / Default: no overlap
#    NormalizationType  = None,   # Z-Score-Normalisierung / Z-score normalization
    BatchSize          = 8,
    TestLabels         = TestLabels, # Labels dem Test-Dataset mitgeben / Pass labels to test dataset
)

TranADDataset: 0 Fenster aus 600 Zeitpunkten (W=1000, S=1000) / 0 windows from 600 timesteps
TranADDataset: 0 Fenster aus 200 Zeitpunkten (W=1000, S=1000) / 0 windows from 200 timesteps


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [7]:
from utils.dataset import ECGReconstructionDataset, SimpleDataManager
from torch.utils.data import DataLoader

# Using my Dataset and Dataloader because of Mismatch

In [8]:
# ── 2. Costume DataLoader erstellen ─────────
NumFeatures = TrainingData.shape[-1]   # 12 ECG leads
WindowSize = TrainingData.shape[1]     # 1000 time points


train_dummy_labels = np.zeros(len(TrainingData), dtype=np.int64)
test_dummy_labels = np.zeros(len(TestData), dtype=np.int64)

# inverse-normalization helper
def inverse_normalize(x_np): return x_np * train_std + train_mean


ECG_Train_Dataset = ECGReconstructionDataset(TrainingData,train_dummy_labels)
Training_DataLoader = DataLoader(ECG_Train_Dataset, batch_size=8, shuffle=True)

ECG_Test_Dataset = ECGReconstructionDataset(TestData,test_dummy_labels)
Test_DataLoader = DataLoader(ECG_Test_Dataset, batch_size=8, shuffle=False)

ECGDataManager = SimpleDataManager(Training_DataLoader, Test_DataLoader, ECG_Test_Dataset, WindowSize, TestLabels, inverse_normalize)



In [9]:
from models.tranad import TranADMainModel

In [14]:
# ── 3. TranADMainModel erstellen / Create TranADMainModel ───────────
TranAD = TranADMainModel(
    NumFeatures         = 12,   # Muss mit Daten übereinstimmen / Must match data
    ModelDimension      = 32,            # Kleine Dimension für schnellen Demo-Lauf / Small dim for fast demo
    NumHeads            = 2,             # Zwei Attention-Köpfe / Two attention heads
    NumEncoderLayers    = 2,             # Flacher Encoder / Shallow encoder
    NumDecoderLayers    = 2,             # Flache Decoder / Shallow decoders
    ForwardDimension    = 64,            # d_ff = 2 × d_model / d_ff = 2 × d_model
    DropoutRate         = 0.1,           # 10% Dropout / 10% dropout
    LearningRate        = 5e-4,          # Leicht erhöhte LR für schnellere Konvergenz / Slightly higher LR
    FocusGrowthRate     = 1.3,           # φ wächst 30% pro Epoche / φ grows 30% per epoch
    InitialFocusScore   = 1.0,           # φ startet bei 1 / φ starts at 1
    MaxFocusScore       = 50.0,          # φ maximal 50 / φ maximum 50
    AnomalyPercentile   = 95.0,          # 95. Perzentil als Anomalie-Schwelle / 95th percentile as threshold
)

TranADModel: 68,664 Parameter | d_model=32, H=2, Enc=2, Dec×2=2


In [15]:
# ── 4. Training ──────────────────────────────────────────────────────
print("\n── Training starts ─────────────────────────────")
TranAD.TrainModel(
    TrainingLoader = Training_DataLoader,
    NumEpochs      = 30,    # 30 Epochen für Demo / 30 epochs for demo
    PrintInterval  = 5,     # Fortschrittsausgabe alle 5 Epochen / Progress every 5 epochs
)


── Training starts ─────────────────────────────
Epoche    1/30 | Loss=2.11175 | φ=1.30 | LR=5.00e-04
Epoche    5/30 | Loss=1.43247 | φ=3.71 | LR=5.00e-04
Epoche   10/30 | Loss=3.50562 | φ=13.79 | LR=5.00e-04
Epoche   15/30 | Loss=10.86332 | φ=50.00 | LR=2.50e-04
Epoche   20/30 | Loss=12.85499 | φ=50.00 | LR=2.50e-04
Epoche   25/30 | Loss=12.14444 | φ=50.00 | LR=1.25e-04
Epoche   30/30 | Loss=11.61553 | φ=50.00 | LR=1.25e-04
Training complete.


In [17]:
# ── 5. Anomalie-Erkennung auf Testdaten / Anomaly detection on test data ─
print("\n── Anomaly Detection ──────────────────────────")
AnomalyScores, BinaryPredictions = TranAD.DetectAnomalies(Test_DataLoader)
print(f"Detected anomalous windows: {BinaryPredictions.sum()}")


── Anomaly Detection ──────────────────────────
Detected anomalous windows: 10


In [18]:
# ── 6. Evaluation mit Ground-Truth / Evaluation with ground-truth ────
print("\n── Evaluation ──────────────────────────────────────────────────────")
# Window-Labels ableiten: Fenster gilt als anomal wenn ≥1 Punkt anomal
WindowLabels = np.array([
    TestLabels[i * WindowSize:(i + 1) * WindowSize].max()  # Max = 1 wenn irgendein Punkt anomal / max=1 if any anomalous
    for i in range(len(Test_DataLoader))
])
Metrics = TranAD.Evaluate(Test_DataLoader, WindowLabels)


── Evaluation ──────────────────────────────────────────────────────


ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:

# ── 7. Explainability / Explainability ────────────────────────────────
print("\n── Explainability ──────────────────────────────────────────────────")
Explainer = TranADExplainability(TranAD)   # Explainer mit Modellreferenz / Explainer with model reference

# 7a. Trainingsverlauf: Loss und φ / Training history: loss and φ
Explainer.ShowTrainingHistory()

# 7b. Erstes Testfenster holen / Get first test window
FirstWindow, _ = DataManager.TestDataset[0]               # (T, F)
FirstWindow    = FirstWindow.unsqueeze(0).to(TranAD.Device)  # → (1, T, F)

# 7c. Phase-1 vs. Phase-2 Vergleich für Feature 0 / Phase-1 vs. Phase-2 comparison
Explainer.ShowPhaseComparison(
    FirstWindow, FeatureIndex=0, DatasetForInverseNorm=DataManager.TestDataset
)

# 7d. Rekonstruktionsfehler-Visualisierung / Reconstruction error visualization
with torch.no_grad():
    _, Rec2 = TranAD.Network(FirstWindow)               # Phase-2-Rekonstruktion / Phase-2 reconstruction
OrigNP = DataManager.TestDataset.InverseNormalization(FirstWindow[0].cpu().numpy())  # (T, F) Originalskala
RecNP  = DataManager.TestDataset.InverseNormalization(Rec2[0].cpu().numpy())          # (T, F)

Explainer.ShowReconstructionError(
    OriginalData      = OrigNP,
    ReconstructedData = RecNP,
    AnomalyScores     = AnomalyScores,
    TrueLabels        = WindowLabels,
    Threshold         = TranAD.AnomalyThreshold,
)

# 7e. Attention-Heatmap für Encoder Layer 0 / Attention heatmap for encoder layer 0
Explainer.ShowAttentionHeatmap(FirstWindow, LayerIndex=0, HeadIndex=0, Component="Encoder")

# 7f. Feature-Wichtigkeit mit Saliency / Feature importance with saliency
Explainer.ShowFeatureImportance(
    FirstWindow,
    FeatureNames=["Sinus", "Kosinus", "Oberton", "Drift", "Kreuzprodukt"],
)

# 7g. Focus-Score-Effekt / Focus score effect
Explainer.ShowFocusScoreEffect(FirstWindow, FocusScoreList=[0.5, 1.0, 2.0, 5.0, 10.0, 30.0])

print("\n TranAD demo complete.")